# PT01: Introduction to PyTorch

Read and run this notebook before PT02. Knowledge of NumPy helps; most of the array syntax should look familiar.

We will use a small batch of measurements to compute predictions, a loss, and its gradients. The optional material at the end can wait until after PT02.

All cells are complete. Before running a cell, try to predict its output or at least its shape. If something is unexpected, change a small example and run it again.

In [1]:
import torch
import torch.nn.functional as F

torch.manual_seed(7)
torch.set_printoptions(precision=3, sci_mode=False)
print("PyTorch", torch.__version__)

PyTorch 2.3.0


## 1. Tensors

The core array type is `torch.Tensor`. We can construct one with `torch.tensor`, or use functions such as `torch.zeros` and `torch.randn`.

Here, each row is an example and each column is a feature: $X \sim (B,D)$, with $B=3$ and $D=2$. These are made-up measurements; we will use real data in PT02.

In [2]:
X = torch.tensor([[1., 2.], [3., 1.], [2., 4.]])
print(X)
print("Shape:", X.shape)
print("Dtype:", X.dtype)
print("Device:", X.device)

tensor([[1., 2.],
        [3., 1.],
        [2., 4.]])
Shape: torch.Size([3, 2])
Dtype: torch.float32
Device: cpu


For now everything stays on the CPU. A tensor also has a device, so moving a model to a GPU requires moving its data there too. `X.to(...)` returns a tensor; keep that return value if you want to use it.

Indexing follows NumPy's zero-based convention. Compare the last two expressions below: selecting a column can remove its dimension.

In [3]:
print("Second example:", X[1])
print("First two examples:\n", X[:2])
print("Last feature:", X[:, -1], X[:, -1].shape)
print("Last feature, keeping its axis:\n", X[:, -1:], X[:, -1:].shape)

Second example: tensor([3., 1.])
First two examples:
 tensor([[1., 2.],
        [3., 1.]])
Last feature: tensor([2., 1., 4.]) torch.Size([3])
Last feature, keeping its axis:
 tensor([[2.],
        [1.],
        [4.]]) torch.Size([3, 1])


## 2. A batch of linear predictions

For one example, our model is $z=Wx+b$. With examples stored in rows, the batched version is

$$Z=XW^\top+b.$$

Here $W\sim(C,D)$ and $Z\sim(B,C)$. The bias has shape $(C,)$ and is broadcast over the examples. `@` is matrix multiplication; `*` is elementwise multiplication.

In [4]:
W = torch.tensor([[1., -1.], [0.5, 2.]])  # (C, D)
b = torch.tensor([0.2, -0.3])             # (C,)
Z = X @ W.T + b                         # (B, C)
print(Z)
torch.testing.assert_close(Z[0], W @ X[0] + b)

tensor([[-0.800,  4.200],
        [ 2.200,  3.200],
        [-1.800,  8.700]])


Reductions remove an axis unless we ask to keep it. This matters when the result is used in another broadcast.

For example, subtracting the mean of each feature requires reducing over the examples, not over the features. Try changing `dim=0` to `dim=1` below and inspect what gets centered.

In [5]:
feature_mean = X.mean(dim=0, keepdim=True)
centered = X - feature_mean
print("Mean shape:", feature_mean.shape)
print(centered)
torch.testing.assert_close(centered.mean(dim=0), torch.zeros(2), atol=1e-6, rtol=0)

Mean shape: torch.Size([1, 2])
tensor([[-1.000, -0.333],
        [ 1.000, -1.333],
        [ 0.000,  1.667]])


## 3. Computing gradients

Consider a regression model $\hat y=Xw+b$ with a mean squared error. We tell PyTorch which parameters need gradients using `requires_grad=True`. The data do not need gradients to train the parameters.

Before running the next cells, predict the shapes of the loss, `w.grad` and `bias.grad`.

In [6]:
y = torch.tensor([1., -1., 2.])
w = torch.tensor([0.2, -0.1], requires_grad=True)
bias = torch.tensor(0., requires_grad=True)

prediction = X @ w + bias
loss = ((prediction - y) ** 2).mean()
print("Loss:", loss.item(), "Shape:", loss.shape)
print("Parameter is a leaf:", w.is_leaf)
print("Prediction is a leaf:", prediction.is_leaf)
loss.backward()
print("w.grad:", w.grad)
print("bias.grad:", bias.grad)

Loss: 2.4166667461395264 Shape: torch.Size([])
Parameter is a leaf: True
Prediction is a leaf: False
w.grad: tensor([-0.333, -5.667])
bias.grad: tensor(-1.)


`backward()` computes gradients of this scalar loss. By default, `.grad` is populated for the leaf tensors that require gradients, such as `w` and `bias`. An intermediate tensor can require gradients without storing them in its own `.grad` field.

PyTorch records operations during the forward pass. Backward uses the saved information and normally releases the saved tensors afterward. To compute gradients for a new step, run a new forward pass. Reusing this same `loss` for another backward pass would require retaining its graph, which we do not need here.

## 4. Updating the parameters

Recall the update $\theta\leftarrow\theta-\eta\nabla L(\theta)$. Updating parameters is bookkeeping for the next step, so we perform it inside `torch.no_grad()`.

We then clear the gradients. `backward()` **adds** to existing gradients; it does not overwrite them.

In [7]:
lr = 0.01
before = loss.item()
with torch.no_grad():
    w -= lr * w.grad
    bias -= lr * bias.grad
w.grad = None
bias.grad = None

after = ((X @ w + bias - y) ** 2).mean()
print(f"Loss before: {before:.4f}; after one update: {after.item():.4f}")
assert after.item() < before  # This example uses a sufficiently small step.

Loss before: 2.4167; after one update: 2.1115


We can see accumulation without training anything. Keep the parameters fixed and differentiate the same function twice, rebuilding the forward computation each time.

In [8]:
a = torch.tensor([1., 2.], requires_grad=True)
a.square().sum().backward()
first_gradient = a.grad.clone()
a.square().sum().backward()
print("First backward:", first_gradient)
print("Second backward:", a.grad)
torch.testing.assert_close(a.grad, 2 * first_gradient)
a.grad = None

First backward: tensor([2., 4.])
Second backward: tensor([4., 8.])


For logging, `loss.item()` gives a Python number. `tensor.detach()` gives a tensor disconnected from its gradient history. Use these when storing results that will not be differentiated later.

`torch.autograd.grad` is another interface: it returns gradients directly instead of accumulating them into `.grad`. We will use it again when studying autodiff.

In [9]:
a = torch.tensor([1., 2.], requires_grad=True)
(gradient,) = torch.autograd.grad(a.square().sum(), a)
print(gradient)
assert a.grad is None

tensor([2., 4.])


## 5. Classification uses the same steps

For classification, $Z\sim(B,C)$ contains logits. The target $y\sim(B,)$ contains integer class indices. Softmax turns logits into probabilities when we need to inspect them.

PyTorch's cross-entropy function takes **logits directly**. It combines the logarithm and normalization in a numerically stable form. Do not pass the probabilities to it.

In [10]:
logits = torch.tensor([[2., 0., -1.], [-1., 0., 2.]], requires_grad=True)
labels = torch.tensor([0, 2], dtype=torch.long)
classification_loss = F.cross_entropy(logits, labels)
classification_loss.backward()
print("Probabilities:\n", logits.detach().softmax(dim=-1))
print("Loss:", classification_loss.item())
print("Gradient shape:", logits.grad.shape)

Probabilities:
 tensor([[0.844, 0.114, 0.042],
        [0.042, 0.114, 0.844]])
Loss: 0.1698460429906845
Gradient shape: torch.Size([2, 3])


For an ordinary training step we will clear gradients, compute predictions and a scalar loss, call `backward()`, then update the parameters. Evaluation needs predictions but does not need to build their gradient history.

`model.eval()` and `torch.no_grad()` do different things. The first changes the behavior of layers such as dropout and batch normalization. The second disables gradient recording. In PT02 we will use both when evaluating a model.

Before PT02, make sure you can explain the bias broadcast, the shapes of the gradients, and what happens if we forget to clear them. The remaining examples are optional.

## Optional: views, copies and in-place operations

Some indexing operations return a view, while others return a copy. A small mutation makes the distinction visible without inspecting storage pointers.

In [11]:
A = torch.arange(6.).reshape(3, 2)
view = A[:2]
copy = A[[0, 1]]
view[0, 0] = -10
print("Original:\n", A)
print("Copy made before the update:\n", copy)
assert A[0, 0].item() == -10 and copy[0, 0].item() == 0

# An underscore usually marks an in-place operation.
A.add_(1)
print(A)

Original:
 tensor([[-10.,   1.],
        [  2.,   3.],
        [  4.,   5.]])
Copy made before the update:
 tensor([[0., 1.],
        [2., 3.]])
tensor([[-9.,  2.],
        [ 3.,  4.],
        [ 5.,  6.]])


In-place changes deserve more care when autograd is involved: backward may need a value that the operation overwrites. The explicit parameter update above is a controlled use after backward.

For dense strided tensors, a transpose changes the strides without moving the underlying data. `reshape` returns a view when possible and otherwise copies. We will return to layout when it matters for performance.

In [12]:
A = torch.arange(6.).reshape(3, 2)
print("A:", A.shape, A.stride(), A.is_contiguous())
print("A.T:", A.T.shape, A.T.stride(), A.T.is_contiguous())
print("Flattened transpose:", A.T.reshape(-1))

A: torch.Size([3, 2]) (2, 1) True
A.T: torch.Size([2, 3]) (1, 2) False
Flattened transpose: tensor([0., 2., 4., 1., 3., 5.])


## Optional: a few array operations

These examples collect the short exercises from the original notebook. You can try writing each function before reading its implementation.

First, sum the main diagonal and the antidiagonal. For an odd-sized matrix, count the center once in each diagonal, so it contributes twice to the sum.

In [13]:
def sum_diagonals(A):
    return A.diagonal().sum() + A.flip(dims=[1]).diagonal().sum()

A = torch.arange(1., 10.).reshape(3, 3)
torch.testing.assert_close(sum_diagonals(A), torch.tensor(30.))
print(sum_diagonals(A))

tensor(30.)


Next, normalize each row to have mean zero and variance one. Here variance means the average squared deviation (`correction=0`). A constant row becomes zero; there is no way to give it unit variance by rescaling its deviations.

In [14]:
def normalize_rows(A, eps=1e-8):
    centered = A - A.mean(dim=1, keepdim=True)
    std = A.std(dim=1, correction=0, keepdim=True)
    return centered / std.clamp_min(eps)

A = torch.tensor([[1., 2., 4.], [4., 4., 4.]])
normalized = normalize_rows(A)
print(normalized)
torch.testing.assert_close(normalized.mean(dim=1), torch.zeros(2), atol=1e-6, rtol=0)
torch.testing.assert_close(normalized.var(dim=1, correction=0), torch.tensor([1., 0.]))

tensor([[-1.069, -0.267,  1.336],
        [ 0.000,  0.000,  0.000]])


Finally, compute all pairwise cosine similarities. Normalize the rows and multiply by the transpose. This operation will return when we discuss representations.

Cosine similarity is undefined for a zero vector. For this implementation we assign zero to comparisons involving a zero vector, matching PyTorch's normalization convention below.

In [15]:
def cosine_similarity(A):
    unit = F.normalize(A, p=2, dim=1, eps=1e-12)
    return unit @ unit.T

A = torch.tensor([[1., 0.], [0., 1.], [-1., 0.], [0., 0.]])
expected = torch.tensor([[1., 0., -1., 0.], [0., 1., 0., 0.],
                         [-1., 0., 1., 0.], [0., 0., 0., 0.]])
torch.testing.assert_close(cosine_similarity(A), expected)
print(cosine_similarity(A))

tensor([[ 1.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.],
        [-1.,  0.,  1.,  0.],
        [ 0.,  0.,  0.,  0.]])


## Further reading

- [PyTorch tensor tutorial](https://docs.pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)
- [Autograd mechanics](https://docs.pytorch.org/docs/stable/notes/autograd.html)
- [Tensor views](https://docs.pytorch.org/docs/stable/tensor_view.html)

Profiling, `torch.compile`, and traversing backward nodes are outside the required preparation. We will revisit the computation graph in the autodiff material and performance in the systems material.